In [13]:
from dynamicTasteDistortion.dataset_loader import read_ml_raw,  read_yelp_raw, process_ml_df, process_df, read_globo_dataset_raw, process_globo_df
from dynamicTasteDistortion.scripts.data_utils import preprocess_genres
import pandas as pd

In [2]:
def get_core_recsys_statistics(dataset):
    num_users = dataset['user'].nunique()
    num_items = dataset['item'].nunique()
    unique_genres = set(g for genres in dataset['genres'] for g in genres)
    num_genres = len(unique_genres)

    possible_interactions = num_users * num_items
    observed_interactions = len(dataset)
    sparsity = 1 - (observed_interactions / possible_interactions) if possible_interactions > 0 else None

    return {
        'num_users': num_users,
        'num_items': num_items,
        'num_genres': num_genres,
        'sparsity': sparsity
    }


def get_recsys_statistics(dataset, rating_col='binarized_rating'):
    stats = get_core_recsys_statistics(dataset)

    if rating_col in dataset.columns:
        num_positive = (dataset[rating_col] == 1).sum()
        num_negative = (dataset[rating_col] == 0).sum()
        total = num_positive + num_negative
        stats['positive_ratio'] = num_positive / total if total > 0 else 0
    else:
        stats['positive_ratio'] = None

    return stats

In [15]:
globo = read_globo_dataset_raw()

Skipping: 'dynamicTasteDistortion/data/globo' already exists and contains files.
Download finished! Merging intermediate files.


Processing files...: 100%|██████████| 385/385 [00:47<00:00,  8.08it/s]


In [26]:
from dynamicTasteDistortion.simulationConstants import GLOBO_PATH


globo_articles = pd.read_csv(f"{GLOBO_PATH}/articles_metadata.csv")[
    ["article_id", "category_id"]
]

In [16]:
globo_processed = process_globo_df(globo)

Enriching globo.com dataset with k=3 random negative samples, due to implcit feedback


Processing users...: 100%|██████████| 1977426/1977426 [00:02<00:00, 895284.44it/s]


In [3]:
ml_1m = read_ml_raw("s")

Skipping: 'dynamicTasteDistortion/data/movielens/raw/ml_s/' already exists and contains files.
Joining movies data with ratings data...


In [4]:
ml_1m_cleaned = process_ml_df(ml_1m)

Preprocessing dataset...


In [9]:
yelp = read_yelp_raw(1000000)

Skipping: 'dynamicTasteDistortion/data/yelp/raw/yelp_1000000/' already exists and contains files.


In [10]:
yelp_cleaned = process_df(yelp)

Preprocessing dataset...


In [11]:
yelp_cleaned

,user,item,rating,date,genres,binarized_rating,timestamp
0,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3.0,2018-07-07 22:09:11,"Restaurants, Breakfast & Brunch, Food, Juice B...",0,1531001351
1,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3.0,2014-02-05 20:30:30,"Restaurants, Breakfast & Brunch",0,1391632230
2,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4.0,2017-01-14 20:54:15,"Sandwiches, Beer, Wine & Spirits, Bars, Food, ...",1,1484427255
3,r3zeYsv1XFBRA4dJpL78cw,gmjsEdUsKpj9Xxu6pdjH0g,5.0,2015-01-03 23:21:18,"Supernatural Readings, Tours, Hotels & Travel,...",1,1420327278
4,1WHRWwQmZOZDAhp2Qyny4g,uMvVYRgGNXf5boolA9HXTw,5.0,2015-06-21 14:48:06,"Shopping, Jewelry",1,1434898086
...,...,...,...,...,...,...,...
465282,RypZ2P165GB_igLo2JXw9Q,lnEeMas32ddqNn06jOUGgQ,1.0,2018-04-21 22:16:51,"Hotels, Hotels & Travel, Event Planning & Serv...",0,1524349011
465283,HBC6FIBHxo40h01tNhxn2g,fp7IHO1z53wYP4PzKSl11g,3.0,2011-11-13 21:35:09,"Food, Specialty Food, Korean, Chinese, Ethnic ...",0,1321220109
465284,i6L9I8xDF-di3kgvDewnjg,efGqAYPCCxPvHPX3dCM8bQ,5.0,2015-02-22 22:53:08,"Arts & Entertainment, Music Venues, Nightlife,...",1,1424645588
465285,oX7o1TH0PHUWp9r9ry9_vw,jLn69WQupjsDKrbPw_nlGQ,3.0,2017-11-15 09:43:07,"Ice Cream & Frozen Yogurt, Food",0,1510738987


In [12]:
yelp_cleaned["genres"] = preprocess_genres(yelp_cleaned, "genres", SEP=",")

In [ ]:
# Number of users, items, unique genres for ml_1m
num_users_ml = ml_1m['user'].nunique()
num_items_ml = ml_1m['item'].nunique()
unique_genres_ml = set(g for genres in ml_1m['genres'] for g in genres)
num_genres_ml = len(unique_genres_ml)

# Number of users, items, unique genres for yelp
num_users_yelp = yelp['user'].nunique()
num_items_yelp = yelp['item'].nunique()
unique_genres_yelp = set(g for genres in yelp_cleaned['genres'] for g in genres)
num_genres_yelp = len(unique_genres_yelp)


# ML-1M sparsity
ml_possible = num_users_ml * num_items_ml
ml_observed = len(ml_1m)
ml_sparsity = 1 - (ml_observed / ml_possible)

# Yelp sparsity
yelp_possible = num_users_yelp * num_items_yelp
yelp_observed = len(yelp)
yelp_sparsity = 1 - (yelp_observed / yelp_possible)

print("ML-1M: sparsity =", ml_sparsity)
print("Yelp: sparsity =", yelp_sparsity)


print("ML-1M: users =", num_users_ml, ", items =", num_items_ml, ", genres =", num_genres_ml)
print("Yelp: users =", num_users_yelp, ", items =", num_items_yelp, ", genres =", num_genres_yelp)

ML-1M: sparsity = 0.9553163743776871
Yelp: sparsity = 0.9998598761598543
ML-1M: users = 6040 , items = 3706 , genres = 30
Yelp: users = 136356 , items = 24352 , genres = 1149


In [ ]:
# Number of users, items, unique genres for ml_1m
num_users_ml = ml_1m_cleaned['user'].nunique()
num_items_ml = ml_1m_cleaned['item'].nunique()
unique_genres_ml = set(g for genres in ml_1m_cleaned['genres'] for g in genres)
num_genres_ml = len(unique_genres_ml)

# Number of users, items, unique genres for yelp
num_users_yelp = yelp_cleaned['user'].nunique()
num_items_yelp = yelp_cleaned['item'].nunique()
unique_genres_yelp = set(g for genres in yelp_cleaned['genres'] for g in genres)
num_genres_yelp = len(unique_genres_yelp)


# ML-1M sparsity
ml_possible = num_users_ml * num_items_ml
ml_observed = len(ml_1m_cleaned)
ml_sparsity = 1 - (ml_observed / ml_possible)

# Yelp sparsity
yelp_possible = num_users_yelp * num_items_yelp
yelp_observed = len(yelp_cleaned)
yelp_sparsity = 1 - (yelp_observed / yelp_possible)

print("ML-1M: sparsity =", ml_sparsity)
print("Yelp: sparsity =", yelp_sparsity)


print("ML-1M: users =", num_users_ml, ", items =", num_items_ml, ", genres =", num_genres_ml)
print("Yelp: users =", num_users_yelp, ", items =", num_items_yelp, ", genres =", num_genres_yelp)

In [19]:
yelp

,user,item,rating,date,genres
0,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3.0,2018-07-07 22:09:11,"Restaurants, Breakfast & Brunch, Food, Juice B..."
1,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3.0,2014-02-05 20:30:30,"Restaurants, Breakfast & Brunch"
2,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4.0,2017-01-14 20:54:15,"Sandwiches, Beer, Wine & Spirits, Bars, Food, ..."
3,r3zeYsv1XFBRA4dJpL78cw,gmjsEdUsKpj9Xxu6pdjH0g,5.0,2015-01-03 23:21:18,"Supernatural Readings, Tours, Hotels & Travel,..."
4,1WHRWwQmZOZDAhp2Qyny4g,uMvVYRgGNXf5boolA9HXTw,5.0,2015-06-21 14:48:06,"Shopping, Jewelry"
...,...,...,...,...,...
465282,RypZ2P165GB_igLo2JXw9Q,lnEeMas32ddqNn06jOUGgQ,1.0,2018-04-21 22:16:51,"Hotels, Hotels & Travel, Event Planning & Serv..."
465283,HBC6FIBHxo40h01tNhxn2g,fp7IHO1z53wYP4PzKSl11g,3.0,2011-11-13 21:35:09,"Food, Specialty Food, Korean, Chinese, Ethnic ..."
465284,i6L9I8xDF-di3kgvDewnjg,efGqAYPCCxPvHPX3dCM8bQ,5.0,2015-02-22 22:53:08,"Arts & Entertainment, Music Venues, Nightlife,..."
465285,oX7o1TH0PHUWp9r9ry9_vw,jLn69WQupjsDKrbPw_nlGQ,3.0,2017-11-15 09:43:07,"Ice Cream & Frozen Yogurt, Food"


In [21]:
import pandas as pd

def safe_genres(value, sep=","):
    if pd.isna(value) or value is None:
        return []
    if isinstance(value, list):
        return value
    return [g.strip() for g in str(value).split(sep) if str(g).strip()]

yelp["genres"] = yelp["genres"].apply(safe_genres)

In [27]:
globo_raw_with_genres = globo.merge(globo_articles, on="article_id")

In [28]:
globo_raw_with_genres

,user_id,article_id,click_timestamp,user,item,category_id
0,0,157541,1506826828020,0,157541,281
1,0,68866,1506826858020,0,68866,136
2,1,235840,1506827017951,1,235840,375
3,1,96663,1506827047951,1,96663,209
4,2,119592,1506827090575,2,119592,247
...,...,...,...,...,...,...
2988176,10051,84911,1508211557302,10051,84911,174
2988177,322896,30760,1508211672520,322896,30760,26
2988178,322896,157507,1508211702520,322896,157507,281
2988179,123718,234481,1508211513583,123718,234481,375


In [30]:
globo

,user_id,article_id,click_timestamp,user,item
0,0,157541,1506826828020,0,157541
1,0,68866,1506826858020,0,68866
2,1,235840,1506827017951,1,235840
3,1,96663,1506827047951,1,96663
4,2,119592,1506827090575,2,119592
...,...,...,...,...,...
2988176,10051,84911,1508211557302,10051,84911
2988177,322896,30760,1508211672520,322896,30760
2988178,322896,157507,1508211702520,322896,157507
2988179,123718,234481,1508211513583,123718,234481


In [ ]:
globo_raw_with_genres["user"] = globo_raw_with_genres["user_id"]
globo_raw_with_genres["item"] = globo_raw_with_genres["article_id"]
globo_raw_with_genres["genres"] = globo_raw_with_genres["category_id"].astype(str)

AttributeError: Can only use .str accessor with string values!

In [32]:



ml_stats_raw = get_recsys_statistics(ml_1m)
yelp_stats_raw = get_recsys_statistics(yelp)
globo_stats_raw = get_recsys_statistics(globo_raw_with_genres)

# Combine into a dataframe
stats_raw_df = pd.DataFrame({
    'ML-1M': ml_stats_raw,
    'Yelp': yelp_stats_raw,
    'Globo': globo_stats_raw
}).T

stats_raw_df

TypeError: 'int' object is not iterable

In [17]:



ml_stats = get_recsys_statistics(ml_1m_cleaned)
yelp_stats = get_recsys_statistics(yelp_cleaned)
globo_stats = get_recsys_statistics(globo_processed)

# Combine into a dataframe
stats_df = pd.DataFrame({
    'ML-1M': ml_stats,
    'Yelp': yelp_stats,
    'Globo': globo_stats
}).T

stats_df

,num_users,num_items,num_genres,sparsity,positive_ratio
ML-1M,5289.0,3701.0,18.0,0.949831,0.573876
Yelp,136351.0,24344.0,1149.0,0.999860,0.704968
Globo,77239.0,31227.0,154.0,0.996724,NaN


In [ ]:

# cleaned datasets
ml_stats = get_recsys_statistics(ml_1m_cleaned)
yelp_stats = get_recsys_statistics(yelp_cleaned)
globo_stats = get_recsys_statistics(globo_processed)

# Combine into a dataframe
stats_df = pd.DataFrame({
    'ML-1M': ml_stats,
    'Yelp': yelp_stats,
    'Globo': globo_stats
}).T

stats_df

## Boostrapped sets

In [3]:
from dynamicTasteDistortion.simulationConstants import SIMULATION_PATH
import pickle 

In [4]:
def load_bootstrapped_clicks_ipynb(data_type, file_size, num_users):
    output_path = (
    f"{SIMULATION_PATH}/{data_type}_{file_size}_n_users={num_users}_bootstrapped.pkl"
)
    with open(output_path, "rb") as f:
        bootstrapped_clicks = pickle.load(f)
    return bootstrapped_clicks


In [10]:
!pwd

/home/caio/dev/dynamicTasteDistortion/dynamicTasteDistortion


In [5]:
from dynamicTasteDistortion.ioUtils import load_bootstrapped_clicks

file_size = "1m"
num_users = 1000
data_type = "ml"

ml_boots = load_bootstrapped_clicks("ml", file_size, num_users)
# output_path = (
#     f"./{SIMULATION_PATH}/{data_type}_{file_size}_n_users={num_users}_bootstrapped.pkl"
# )
# with open(output_path, "rb") as f:
#     bootstrapped_clicks = pickle.load(f)

In [13]:
from dynamicTasteDistortion.ioUtils import get_timestamp_behavior_path


pd.read_csv(
        get_timestamp_behavior_path(data_type, file_size, num_users)
    )

,user,median_timestamp_diff,std_timestamp_diff,n_entries
0,0,20.0,5.237711e+01,51
1,1,17.0,2.605920e+01,211
2,2,14.0,6.877114e+05,136
3,3,17.0,2.849323e+01,70
4,4,15.0,1.758540e+06,351
...,...,...,...,...
995,995,17.0,4.891840e+03,271
996,996,17.0,2.611830e+01,31
997,997,17.0,2.486708e+01,115
998,998,19.5,5.507059e+01,51


In [9]:
yelp_boots = load_bootstrapped_clicks("yelp", file_size, num_users)

In [11]:
yelp_boots

,user,item,relevant,clicked_at,timestamp,rating,constant,linear_time,genres
0,0,242,1.0,0,1.177022e+05,8.533658e-01,1,2.456351e-03,"(food, beer, wine-&-spirits, nightlife, restau..."
1,0,1197,1.0,1,6.670199e+04,4.094931e-01,1,5.972482e-04,"(bars, beer-gardens, food, breweries, nightlif..."
2,0,556,1.0,2,6.532069e+04,5.571659e-01,1,7.912026e-04,"(restaurants, food, chinese, asian-fusion, des..."
3,0,2034,1.0,3,1.674200e+05,6.751899e-01,1,2.878091e-03,"(sandwiches, american-(new), lounges, nightlif..."
4,0,376,1.0,4,3.911916e+04,1.000000e-08,1,6.905605e-12,"(food-stands, food, hot-dogs, street-vendors, ..."
...,...,...,...,...,...,...,...,...,...
500558,998,1807,1.0,12,1.710128e+08,8.043002e-01,1,8.009667e-01,"(beer, wine-&-spirits, food, bars, nightlife)"
500559,998,1359,1.0,18,1.712423e+08,5.129210e-01,1,5.114810e-01,"(food, restaurants, seafood)"
500560,998,2149,1.0,52,1.717241e+08,1.000000e-08,1,1.000000e-08,"(parks, swimming-pools, boating, fishing, acti..."
500561,999,1989,1.0,0,2.734566e+07,1.000000e-08,1,1.000000e-08,"(active-life, gyms, interval-training-gyms, tr..."


In [8]:
get_recsys_statistics(ml_boots)

{'num_users': 1000,
 'num_items': 3147,
 'num_genres': 18,
 'sparsity': 0.840930409914204,
 'positive_ratio': None}

In [10]:
get_recsys_statistics(yelp_boots)

{'num_users': 976,
 'num_items': 2226,
 'num_genres': 576,
 'sparsity': 0.7695993143623054,
 'positive_ratio': None}